# YouTube Summarizer — Backend Server
Run this notebook on Kaggle (GPU) or Colab to expose the BART model via ngrok.
The Streamlit app will call this server instead of loading the model locally.

In [ ]:
!pip install -q flask flask-cors pyngrok

In [ ]:
import torch
from transformers import BartTokenizer, BartForConditionalGeneration

MODEL_NAME = "facebook/bart-large-cnn"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading {MODEL_NAME} on {DEVICE.upper()} ...")
tokenizer = BartTokenizer.from_pretrained(MODEL_NAME)
model = BartForConditionalGeneration.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()
print("Model ready.")

In [ ]:
from flask import Flask, request, jsonify
from flask_cors import CORS
import threading

app = Flask(__name__)
CORS(app)


def chunk_text(text: str, max_words: int = 300):
    """Split text into word-limited chunks."""
    words = text.split()
    return [
        " ".join(words[i : i + max_words])
        for i in range(0, len(words), max_words)
    ]


def summarize_chunk(text: str, max_length: int, min_length: int) -> str:
    """Tokenize one chunk and run BART generation."""
    inputs = tokenizer(
        text,
        max_length=1024,
        truncation=True,
        return_tensors="pt",
    ).to(DEVICE)

    with torch.no_grad():
        ids = model.generate(
            inputs["input_ids"],
            max_length=max_length,
            min_length=min_length,
            length_penalty=2.0,
            num_beams=4,
            early_stopping=True,
        )

    return tokenizer.decode(ids[0], skip_special_tokens=True)


@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok", "model": MODEL_NAME, "device": DEVICE})


@app.route("/summarize", methods=["POST"])
def summarize():
    """Expects JSON: {text, max_length, min_length}"""
    data = request.get_json(force=True)

    text       = data.get("text", "")
    max_length = int(data.get("max_length", 120))
    min_length = int(data.get("min_length", 40))

    if not text.strip():
        return jsonify({"error": "No text provided"}), 400

    chunks    = chunk_text(text)
    summaries = [summarize_chunk(chunk, max_length, min_length) for chunk in chunks]

    return jsonify({"summaries": summaries, "chunks": len(chunks)})


# Run Flask in a background thread so the cell doesn't block
server_thread = threading.Thread(
    target=lambda: app.run(host="0.0.0.0", port=5000, use_reloader=False)
)
server_thread.daemon = True
server_thread.start()
print("Flask server running on port 5000")

In [ ]:
from pyngrok import ngrok, conf

# ── Paste your ngrok authtoken here ──────────────────────────────────────────
NGROK_AUTH_TOKEN = "" #Put your ngrok api key here
# ─────────────────────────────────────────────────────────────────────────────

conf.get_default().auth_token = NGROK_AUTH_TOKEN

# Kill any existing tunnels before opening a new one
ngrok.kill()

tunnel = ngrok.connect(5000, "http")
public_url = tunnel.public_url

print("\n" + "=" * 60)
print(f"  ngrok public URL: {public_url}")
print("=" * 60)
print("\nCopy the URL above and paste it into the Streamlit app.")
print(f"Health check: {public_url}/health")

### Keep this cell running
The tunnel stays alive as long as this notebook is running.  
Copy the `ngrok public URL` printed above into the Streamlit sidebar.